# Part02 프롬프트와 출력 파서
## CH05 프롬프트

In [ ]:
# pip 어떤 버전이 어떤 가상환경에 설치되어있는지 확인
#!pip --version

pip 26.2.1 from D:\hykim\hanwha_0902\.venv\Lib\site-packages\pip (python 3.12)



In [ ]:
# 패키지 : 설치한 사용라이브러리 묶음
# !pip install python-dotenv

In [ ]:
# LangSmith 추적을 설정합니다.
#!pip install -U langchain langchain-openai

In [ ]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv
import os

# API KEY 정보로드
# .env 파일 로드 (override=False가 기본값이므로 시스템 환경변수가 우선순위를 가집니다)
load_dotenv()

# 정상 로드 확인 테스트
# 설정된 키값을 출력: 키값이 노출되지 않도록 앞부분(8글자만) 출력해본다.
print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH_API_KEY:", os.getenv("LANGSMITH_API_KEY")[:8]+"...")
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT"))
print("LANGSMITH_ENDPOINT:", os.getenv("LANGSMITH_ENDPOINT"))

OPENAI_API_KEY: sk-proj-...
LANGSMITH_API_KEY: lsv2_pt_...
LANGSMITH_PROJECT: hanwha_0902
LANGSMITH_ENDPOINT: https://api.smith.langchain.com


### 01. 프롬프트 템플릿만들기


`PromptTemplate`

- 사용자의 입력 변수를 사용하여 완전한 프롬프트 문자열을 만드는 데 사용되는 템플릿입니다
- 사용법
  - `template`: 템플릿 문자열입니다. 이 문자열 내에서 중괄호 `{}`는 변수를 나타냅니다.
  - `input_variables`: 중괄호 안에 들어갈 변수의 이름을 리스트로 정의합니다.

`input_variables`

- input_variables는 PromptTemplate에서 사용되는 변수의 이름을 정의하는 리스트입니다.

In [123]:
#랭스미스 추척하기.
#logging.langsmith("hanwha_0902", set_enable=True)

#랭스미스 추척하지 않기.
#logging.langsmith("hanwha_0902", set_enable=False)

In [124]:
import logging
from langchain_teddynote import logging
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from langchain_teddynote.messages import stream_response
from langchain_openai import ChatOpenAI

In [125]:
logging.langsmith("hanwha_0902")

LangSmith 추적을 시작합니다.
[프로젝트명]
hanwha_0902


In [126]:
# LLM 객체 정의
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
)

#### 방법 1. from_template() 메소드를 사용하여 PromptTemplate 객체 생성

- 치환될 변수를 `{ 변수 }` 로 묶어서 템플릿을 정의합니다.

In [81]:
from langchain_core.prompts import PromptTemplate

#templete 정의.{python_module}는 변수로, 이후에 값이 들어갈 자리를 의미
template = "파이썬에서 {python_module} 어떻게 사용되나요?"

# from_template 메소드를 이용하여 PromptTemplate 객체 생성
prompt = PromptTemplate.from_template(template)
prompt


PromptTemplate(input_variables=['python_module'], input_types={}, partial_variables={}, template='파이썬에서 {python_module} 어떻게 사용되나요?')

`python_modue` 변수에 값을 넣어서 문장을 생성할 수 있습니다.

In [82]:
# prompt 생성. format 메소드를 이용하여 변수에 값을 넣어줌
prompt = prompt.format(python_module="FastAPI")
prompt

'파이썬에서 FastAPI 어떻게 사용되나요?'

In [ ]:
# template 정의
template = "파이썬에서 {python_module} 어떻게 사용되나요?"

# from_template 메소드를 이용하여 PromptTemplate 객체 생성
prompt = PromptTemplate.from_template(template)

# chain 생성
chain = prompt | llm


In [87]:
# 변수에 입력된 값이 자동으로 치환되어 실행됨
chain.invoke("FastAPI").content

'FastAPI는 Python으로 웹 API를 빠르고 쉽게 구축할 수 있도록 도와주는 현대적인 웹 프레임워크입니다. FastAPI는 비동기 프로그래밍을 지원하며, 자동으로 OpenAPI 문서를 생성해주는 기능이 있어 RESTful API를 만들기에 적합합니다. 아래는 FastAPI를 사용하는 기본적인 방법에 대한 단계별 안내입니다.\n\n### 1. FastAPI 설치\n\nFastAPI와 ASGI 서버인 `uvicorn`을 설치합니다. 터미널에서 다음 명령어를 실행하세요.\n\n```bash\npip install fastapi uvicorn\n```\n\n### 2. 기본 FastAPI 애플리케이션 생성\n\n아래는 FastAPI 애플리케이션의 기본 구조입니다. `main.py`라는 파일을 생성하고 다음 코드를 작성합니다.\n\n```python\nfrom fastapi import FastAPI\n\napp = FastAPI()\n\n@app.get("/")\ndef read_root():\n    return {"Hello": "World"}\n\n@app.get("/items/{item_id}")\ndef read_item(item_id: int, q: str = None):\n    return {"item_id": item_id, "q": q}\n```\n\n### 3. 애플리케이션 실행\n\n터미널에서 다음 명령어를 실행하여 FastAPI 애플리케이션을 실행합니다.\n\n```bash\nuvicorn main:app --reload\n```\n\n- `main`은 파일 이름 (main.py)\n- `app`은 FastAPI 인스턴스의 이름\n- `--reload` 플래그는 코드 변경 시 자동으로 서버를 재시작합니다.\n\n### 4. API 테스트\n\n브라우저에서 `http://127.0.0.1:8000`에 접속하면 `{"Hello": "World"}`라는 JSON 응답을 받을 수 있습니다. 또한, `http://127.0.0.1:8000/it

#### Python에서 하나의 full code 실행할때
- template을 먼저 정의한 뒤 PromptTemplate을 만들어야 한다.

In [90]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# PromptTemplate 정의
# {python_module}은 실행할 때 값이 들어가는 변수 자리입니다.
template = "파이썬에서 {python_module} 어떻게 사용되나요?"

# template으로 PromptTemplate 객체 생성
prompt = PromptTemplate.from_template(template)

# LLM 객체 정의
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
)

# prompt와 LLM을 연결해 체인 생성
chain = prompt | llm



# "FastAPI"가 {python_module} 자리에 자동으로 들어가 실행됩니다.
chain.invoke("FastAPI").content


'FastAPI는 Python으로 작성된 현대적인 웹 프레임워크로, 빠르고 효율적인 API를 구축하는 데 사용됩니다. FastAPI는 비동기 프로그래밍을 지원하며, 자동으로 OpenAPI 문서를 생성하는 기능이 있습니다. 아래는 FastAPI를 사용하는 기본적인 방법에 대한 단계별 안내입니다.\n\n### 1. FastAPI 설치\n\n먼저 FastAPI와 ASGI 서버인 `uvicorn`을 설치해야 합니다. 다음 명령어를 사용하여 설치할 수 있습니다.\n\n```bash\npip install fastapi uvicorn\n```\n\n### 2. 기본 FastAPI 애플리케이션 생성\n\nFastAPI 애플리케이션을 생성하는 기본적인 예제는 다음과 같습니다.\n\n```python\nfrom fastapi import FastAPI\n\napp = FastAPI()\n\n@app.get("/")\nasync def read_root():\n    return {"Hello": "World"}\n\n@app.get("/items/{item_id}")\nasync def read_item(item_id: int, q: str = None):\n    return {"item_id": item_id, "q": q}\n```\n\n### 3. 애플리케이션 실행\n\n위의 코드를 `main.py`라는 파일에 저장한 후, 다음 명령어로 애플리케이션을 실행할 수 있습니다.\n\n```bash\nuvicorn main:app --reload\n```\n\n- `main`은 파일 이름 (확장자 제외)\n- `app`은 FastAPI 인스턴스의 이름\n- `--reload` 플래그는 코드 변경 시 자동으로 서버를 재시작합니다.\n\n### 4. API 테스트\n\n서버가 실행되면 웹 브라우저에서 `http://127.0.0.1:8000`에 접속하여 "Hello World" 메시지를 확인할 수 있습니다. 또한, `http://127.0.0.1:8000/items/5?q=som

#### 방법 2. PromptTemplate 객체 생성과 동시에 prompt 생성


추가 유효성 검사를 위해 `input_variables` 를 명시적으로 지정하세요.

이러한 변수는 인스턴스화 중에 템플릿 문자열에 있는 변수와 비교하여 불일치하는 경우 예외를 발생시킵니다.

In [110]:
# template 정의
template = "파이썬에서 {python_module} 어떻게 사용되나요?"

# PromptTemplate 객체를 활용하여 prompt_template 생성
prompt = PromptTemplate(
    template=template,
    input_variables=["python_module"],
)

prompt

PromptTemplate(input_variables=['python_module'], input_types={}, partial_variables={}, template='파이썬에서 {python_module} 어떻게 사용되나요?')

In [111]:
# prompt 생성
prompt.format(python_module="FastAPI")

'파이썬에서 FastAPI 어떻게 사용되나요?'

In [112]:
# templat 정의
template = "파이썬에서 {python_module1}과 {python_module2}은 각각 어떻게 사용되나요?"

# PromptTemplate 객체를 활용하여 prompt_template 생성
prompt = PromptTemplate(
    template = template,
    input_variables=["python_module1"],
    partial_variables={
        "python_module2": "Streamlit"  # dictionary 형태로 partial_variables를 전달
    }
)

prompt


PromptTemplate(input_variables=['python_module1'], input_types={}, partial_variables={'python_module2': 'Streamlit'}, template='파이썬에서 {python_module1}과 {python_module2}은 각각 어떻게 사용되나요?')

In [116]:
prompt.format(python_module1="FastAPI")

'파이썬에서 FastAPI과 Streamlit은 각각 어떻게 사용되나요?'

In [117]:
prompt_partial = prompt.partial(python_module2="Streamlit")
prompt_partial

PromptTemplate(input_variables=['python_module1'], input_types={}, partial_variables={'python_module2': 'Streamlit'}, template='파이썬에서 {python_module1}과 {python_module2}은 각각 어떻게 사용되나요?')

In [118]:
prompt_partial.format(python_module1="FastAPI")

'파이썬에서 FastAPI과 Streamlit은 각각 어떻게 사용되나요?'

In [119]:
chain = prompt_partial | llm

In [120]:
chain.invoke("FastAPI").content

'FastAPI와 Streamlit은 각각 웹 애플리케이션을 개발하는 데 사용되는 파이썬 프레임워크입니다. 두 프레임워크는 서로 다른 목적과 사용 사례를 가지고 있습니다.\n\n### FastAPI\n\nFastAPI는 고성능 API를 구축하기 위한 웹 프레임워크입니다. 주로 RESTful API를 만들 때 사용되며, 비동기 프로그래밍을 지원하고, 자동으로 OpenAPI 문서를 생성하는 기능이 있습니다.\n\n#### FastAPI 사용 예제\n\n1. **설치**:\n   ```bash\n   pip install fastapi uvicorn\n   ```\n\n2. **간단한 API 서버 만들기**:\n   ```python\n   from fastapi import FastAPI\n\n   app = FastAPI()\n\n   @app.get("/")\n   def read_root():\n       return {"Hello": "World"}\n\n   @app.get("/items/{item_id}")\n   def read_item(item_id: int, q: str = None):\n       return {"item_id": item_id, "q": q}\n   ```\n\n3. **서버 실행**:\n   ```bash\n   uvicorn main:app --reload\n   ```\n\n4. **API 문서 확인**: \n   기본적으로 `/docs` 경로에서 Swagger UI를 통해 API 문서를 확인할 수 있습니다.\n\n### Streamlit\n\nStreamlit은 데이터 애플리케이션을 쉽게 만들 수 있도록 도와주는 프레임워크입니다. 주로 데이터 시각화와 대시보드 구축에 사용됩니다. 사용자가 인터랙티브한 웹 애플리케이션을 쉽게 만들 수 있도록 설계되었습니다.\n\n#### Streamlit 사용 예제\n\n1. **설치**:\n   ```bash\n   pip install streamlit\n   ```\n\n2. **

In [122]:
chain.invoke({"python_module1": "FastAPI", "python_module2": "@decorator"}).content

'FastAPI와 데코레이터는 Python에서 웹 애플리케이션을 구축할 때 매우 유용하게 사용됩니다. 아래에서 각각의 사용법을 설명하겠습니다.\n\n### FastAPI 사용법\n\nFastAPI는 Python으로 작성된 현대적인 웹 프레임워크로, 빠르고 간편하게 API를 구축할 수 있도록 도와줍니다. FastAPI를 사용하기 위해서는 먼저 설치해야 합니다.\n\n```bash\npip install fastapi uvicorn\n```\n\nFastAPI를 사용하여 간단한 API를 만드는 예제는 다음과 같습니다:\n\n```python\nfrom fastapi import FastAPI\n\napp = FastAPI()\n\n@app.get("/")\nasync def read_root():\n    return {"Hello": "World"}\n\n@app.get("/items/{item_id}")\nasync def read_item(item_id: int, q: str = None):\n    return {"item_id": item_id, "q": q}\n```\n\n위의 코드에서 `@app.get("/")`는 FastAPI의 라우터 데코레이터로, HTTP GET 요청을 처리하는 엔드포인트를 정의합니다. `read_root` 함수는 루트 URL에 대한 요청을 처리하고 JSON 응답을 반환합니다.\n\n### 데코레이터 사용법\n\nPython의 데코레이터는 함수나 메서드의 동작을 수정하거나 확장하는 데 사용되는 함수입니다. 데코레이터는 주로 함수 정의 위에 `@decorator_name` 형식으로 사용됩니다.\n\n예를 들어, 간단한 데코레이터를 만들어 보겠습니다:\n\n```python\ndef my_decorator(func):\n    def wrapper():\n        print("Something is happening before the function is called.")\n        func()\n        print

### 02. 부분 변수 활용하기

`partial`을 사용하는 일반적인 용도는 함수를 부분적으로 사용하는 것입니다. 이 사용 사례는 항상 공통된 방식으로 가져오고 싶은 변수 가 있는 경우입니다.

대표적인 예가 날짜나 시간 입니다.

항상 현재 날짜를 반환하는 함수 를 사용하여 프롬프트를 부분적으로 변경할 수 있으면 매우 편리합니다.